In [231]:
# Autograd = Automatically computes gradients to update weights during training.

In [232]:
# manually writing the derivative of y = x^2
# we know that dy/dx = 2x
def dy_dx(x):
    return(2*x)

In [233]:
dy_dx(3)  # the slope of x^2 at x=3 is 2*3 = 6

6

In [234]:
import torch

# Tell PyTorch to remember this tensor so it can calculate its gradient later
x = torch.tensor(3.0, requires_grad=True)

# Create a new value by squaring x
# PyTorch automatically remembers this operation
y = x ** 2

In [235]:
x  # tensor(3., requires_grad=True) -> gradient tracking is ON

tensor(3., requires_grad=True)

In [236]:
y # tensor(9., grad_fn=<PowBackward0>) -> created by x**2

tensor(9., grad_fn=<PowBackward0>)

## grad_fn=<PowBackward0> means PyTorch remembers that y was created using the power (**2) operation. During backward(), it uses this information to calculate gradients automatically.

In [237]:
# .backward() - compute the gradient (does the backprop/chain rule automatically)
y.backward()  

In [238]:
# .grad - read the computed gradient (the slope)
x.grad       

tensor(6.)

In [239]:
x=torch.tensor(4.0,requires_grad=True)

In [240]:
y=x**2

In [241]:
z=torch.sin(y)

In [242]:
x

tensor(4., requires_grad=True)

In [243]:
y

tensor(16., grad_fn=<PowBackward0>)

In [244]:
z

tensor(-0.2879, grad_fn=<SinBackward0>)

In [245]:
z.backward()

In [246]:
x.grad

tensor(-7.6613)

In [247]:
# checking manually
import math
def dz_dx(x):
    return(2*x*math.cos(x**2))

In [248]:
dz_dx(4.0)

-7.661275842587077

In [249]:
y.grad  
# only LEAF tensors (ones YOU made with requires_grad) store .grad
# y is intermediate (computed from x), so its grad isn't kept (saves memory)

/var/folders/8d/lw8f5qq93psg0982ysksbl8m0000gn/T/ipykernel_5907/1415831939.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:499.)
  y.grad


## Real neural network example 

In [250]:
# a single neuron: input x, true label y, weight w, bias b
x = torch.tensor(6.7)   # input feature
y = torch.tensor(0.0)   # true label (binary: 0 or 1)
w = torch.tensor(1.0)   # weight
b = torch.tensor(0.0)   # bias

In [251]:
# Binary Cross-Entropy loss 
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8   # tiny number(1e-8 = 0.00000001) to prevent log(0) which would be -infinity
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)  # keep away from exactly 0 or 1
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [252]:
# FORWARD PASS (making the prediction)
z = w * x + b               # Step 1: weighted sum
y_pred = torch.sigmoid(z)   # Step 2: sigmoid squashes to 0-1 (probability)
loss = binary_cross_entropy_loss(y_pred, y)   # how wrong was it?

loss 

tensor(6.7012)

In [253]:
# manually applying the chain rule: dL/dw = dL/dy_pred × dy_pred/dz × dz/dw

# how loss changes with the prediction
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# how prediction changes with z
dy_pred_dz = y_pred * (1 - y_pred)

# how z changes with w and b
dz_dw = x   # because z = w*x + b, derivative w.r.t w is x
dz_db = 1   # derivative w.r.t b is 1

# applying chain rule 
dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [254]:
print(f"Manual gradient w.r.t weight: {dL_dw}")   
print(f"Manual gradient w.r.t bias: {dL_db}")     

Manual gradient w.r.t weight: 6.691762447357178
Manual gradient w.r.t bias: 0.998770534992218


## Now the SAME thing with autograd 

In [255]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)
w = torch.tensor(1.0, requires_grad=True)   # track w
b = torch.tensor(0.0, requires_grad=True)   # track b

# forward pass (same as before)
z = w*x + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy_loss(y_pred, y)

# backward pass - autograd computes ALL gradients automatically
loss.backward()

print(w.grad)  
print(b.grad)  

tensor(6.6918)
tensor(0.9988)


In [256]:
w

tensor(1., requires_grad=True)

In [257]:
b

tensor(0., requires_grad=True)

In [258]:
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [259]:
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [260]:
loss

tensor(6.7012, grad_fn=<NegBackward0>)

## Autograd with vector inputs

In [261]:
x=torch.tensor([1.0,2.0,3.0],requires_grad=True)
x

tensor([1., 2., 3.], requires_grad=True)

In [262]:
y=(x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [263]:
y.backward()

In [264]:
x.grad # each element gets its own gradient: d(mean of x^2)/dx_i = 2*x_i/3

tensor([0.6667, 1.3333, 2.0000])

## clearing gradients 

In [265]:
x = torch.tensor(2.0, requires_grad=True)

# first backward
y = x ** 2
y.backward()
print(x.grad)   

# second backward WITHOUT clearing
y=x**2
y.backward()
print(x.grad)     # it added 4+4 instead of resetting

tensor(4.)
tensor(8.)


In [266]:
# grad.zero_() resets the gradient to 0 before each new step
print(x.grad.zero_())  
y = x ** 2
y.backward()
print(x.grad)   

tensor(0.)
tensor(4.)


##  Why this matters: in a real training loop you call .backward() every single iteration. Without clearing, old gradients pile up and corrupt training.

## Disabling gradient tracking (3 ways)
## Sometimes you DON'T want gradient tracking — mainly during testing/prediction (you're not training, so tracking wastes time and memory). Three ways:

In [267]:
x=torch.tensor(2.0,requires_grad=True)
x

tensor(2., requires_grad=True)

In [268]:
# Way 1 — requires_grad_(False) (turn tracking off permanently)
x.requires_grad_(False) 
# now operations on x won't be tracked

tensor(2.)

In [269]:
# Way 2 — .detach() (make an untracked COPY)
z=x.detach() # z is a copy of x WITHOUT tracking
z

tensor(2.)

In [270]:
y=x**2 # tracked (has grad_fn)
y

tensor(4.)

In [271]:
y1=z**2 # NOT tracked (no grad_fn)
y1

tensor(4.)

In [272]:
# Way 3 — torch.no_grad()
x = torch.tensor(2.0, requires_grad=True)

# inside no_grad(), tracking is OFF
with torch.no_grad():
    y = x ** 2
    print(y)     #  NO grad_fn (tracking OFF inside this block)

tensor(4.)


## Why it matters: during testing(making predictions), you don't need gradients — you're not training. torch.no_grad() skips gradient tracking → faster + uses less memory. Every real prediction loop wraps in with torch.no_grad()

In [273]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.1   # learning rate

# 1. FORWARD PASS 
z = w * x + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy_loss(y_pred, y)

# 2. BACKWARD PASS - autograd computes gradients (the chain rule)
loss.backward()

# 3. UPDATE WEIGHTS - gradient descent
with torch.no_grad():
    w -= lr * w.grad     
    b -= lr * b.grad

# 4. CLEAR GRADIENTS - reset for the next step
w.grad.zero_()
b.grad.zero_()

print(w, b)   # the updated weights

tensor(0.3308, requires_grad=True) tensor(-0.0999, requires_grad=True)
